In [ ]:

from collections import deque    #чтоб использовать в топологическом двустороннюю очередь

def topological_sort(v):
    n = len(v)
    g = [[] for a in range(n)]  #смежный список
    indeg = [0] * n           #колво входящих ребер
    for i in range(n - 1):
        g[i].append(i + 1)   #создаем ориентированное ребро
        indeg[i + 1] += 1     #счетчик входящих ребер
    q = deque([i for i in range(n) if indeg[i] == 0])  #создали двустороннюю очереедь
    order = []  #список для вершин в порядке топологической сорт
    while q:
        u = q.popleft()  #взяли номер самой первой вершины
        order.append(u)   
        for nei in g[u]:  #цикл по соседним вершинам
            indeg[nei] -= 1
            if indeg[nei] == 0:
                q.append(nei)   #если нет необработанных соседей, то добавляем в очередь
    return [v[i] for i in order]  

def smooth_sort(v):
    v = v[:]  #копия списка, чтобы не менять исходный
    n = len(v)
    if n < 2:
        return v
    leonardo = [1, 1]  #список чисел леонардо
    while leonardo[-1] < n:
        leonardo.append(leonardo[-1] + leonardo[-2] + 1)

    def sift(i, size):  #функция просейки
        while size > 1:   #пока у кучи есть дети
            r = i - 1   #индекс корня правого поддерева
            l = i - 1 - leonardo[size - 2]  #левого
            if v[l] >= v[r]:
                m = l
                size -= 1
            else:
                m = r
                size -= 2    #ищем индекс максимального поддерева
            if v[m] <= v[i]:   #если нынешний индекс больше максимального поддерева, значит все норм
                break
            v[i], v[m] = v[m], v[i]  #меняем местами их
            i = m

    def trinkle(i, p, size):  #ф-ция подравнивания, т е смотрим еще и корень предыдущей кучи
        while p > 0:  #р у нас тут типо битовая маска
            while p % 2 == 0:
                p //= 2
                size += 1  #слева направо пропускаем кучи, которых нет 
            j = i - leonardo[size]  #корень предыдущей кучи большего размера
            if v[j] <= v[i]:
                break
            v[i], v[j] = v[j], v[i]
            i = j
            p //= 2
            size -= 1
            sift(i, size)  #для восстановления свойства кучи на новой позиции

    q = 1  #индекс следующего элемента для добавления
    size = 1 #размер правой кучи
    p = 1
    while q < n:  #построение леса куч леонардо
        if (p & 3) == 3:   #тк в двоич виде 3=..00011, типо есть две кучи размера 1 вподряд
            sift(q - 1, size)  #просеиваем текущий элемент в только что созданную структуру
            p >>= 2 #сдвигаем на 2 позиции вправо, уюираем 2 кучи наши
            size += 2   #у нас две кучи какбы объединились в одну
        else:
            if leonardo[size - 1] >= n - q:  #размер предыдущей кучи достаточно большой, чтобы вместить оставшиеся элементы
                trinkle(q - 1, p, size)
            else:
                sift(q - 1, size)  #т е создаем новую кучу размера 1 и просеиваем 
            if size == 1:
                p <<= 1   #добавляем 0 в младший бит
                size -= 1
            else:
                p = (p << 1) + 1   #появилась куча размера 1
                size -= 2
        q += 1  #счетчик обработанных элементов
    size -= 1  #переходим на след кучу слева 
    p = (p << 1) + 1  #добавляем информацию о новых маленьких кучках, которые появились при разборке
    while size >= 0:
        trinkle(n - 1, p, size)  #разбираем текущую кучу
        n -= 1  #общий счетчик
        if (p & 1) == 1:
            p >>= 1
            size -= 1  #если есть куча размера 1, то убираем этот бит и переход к след размеру
        else:
            p >>= 2  #если младший юит 0, то пропускаем 2 бита и размер увеличиваетсся
            size += 1
            #те элементы, которые становятся корнями, оказываются на своем окончательномь месте - в конце списка для сортировки по возрастанию
    return v
    
#проверка
data = [5, 2, 8, 3, 1]
print("Исходные данные:", data)
print("Топологическая сортировка:", topological_sort(data))
print("Smoothsort:", smooth_sort(data))
Исходные данные: [5, 2, 8, 3, 1]
Topological sort: [5, 2, 8, 3, 1]
Smoothsort: [2, 5, 8, 3, 1]
 